## Yhteenveto

Logistiikkatoimintojen tiimi suunnittelee satunnaistettua kenttäkoetta, jossa vertaillaan kolmea kuljettajien reititysstrategiaa (staattinen perustaso, dynaaminen uudelleenreititys, AI-optimoitu) kolmella varikkoalueella, vasteena keskimääräinen toimitusviive (minuuttia). PROC GLMPOWER käyttää *esimerkinomaista* aineistoa oletetuista soluarvoista ja ratkaisee kuljettajien kokonaismäärän, joka tarvitaan strategiavaikutusten havaitsemiseen 80 %:n ja 90 %:n voima-arvolla (power), ja kartoittaa sitten, miten saavutettavissa oleva voima-arvo heikkenee reittien välisen vaihtelun kasvaessa.

# Kuljettajien reitityskokeen otoskoon mitoitus PROC GLMPOWER:lla

## Yhteenveto

Logistiikkatoimintojen tiimi on käynnistämässä satunnaistettua kenttäkoetta, jossa vertaillaan kolmea kuljettajien reititysstrategiaa -- **staattista** perustasoa, **dynaamista** uudelleenreititysjärjestelmää ja **AI-optimoitua** suunnittelijaa -- käytössä kolmella varikkoalueella (Northeast, Midwest, West). Vasteena on keskimääräinen **toimitusviive minuutteina**. Ennen kuin tiimi sitoo kalustokapasiteettia kokeeseen, sen täytyy vastata: *kuinka monta kuljettajaa tarvitaan, jotta odotettu toiminnallinen parannus voidaan havaita luotettavasti?*

Tämä muistikirja käyttää **PROC GLMPOWER**:ia kokeen taustalla olevan yleisen lineaarisen mallin prospektiiviseen voima- ja otoskokoanalyysiin. Lähtien *esimerkinomaisesta* aineistosta oletettuja soluarvoja, se (1) ratkaisee kokonaisilmoittautumisen, joka tarvitaan 80% ja 90% voima-arvon saavuttamiseksi kokonaisstrategia- ja aluevaikutuksille, (2) kartoittaa, miten saavutettavissa oleva voima-arvo heikkenee reittien välisen vaihtelun kasvaessa, ja (3) tuottaa voimakäyrän ilmoittautumispäätöksen tueksi.

> **Keskeinen havainto:** Uskottavalla 8 minuutin virheen keskihajonnalla noin **63 kuljettajaa** tuottaa 80% voima-arvon ja **83 kuljettajaa** tuottaa 90% voima-arvon reititysstrategian vaikutusten havaitsemiseen -- mutta jos viiveen vaihtelu on lähempänä 10 minuuttia, jopa 90 ilmoittautunutta kuljettajaa jää alle 70% voima-arvon, mikä korostaa tiukkojen, hyvin instrumentoitujen reittien arvoa.

---
## 1. Esimerkinomaisen asetelman rakentaminen

Ensimmäinen vaihe koodaa kokeen asetelman ja tiimin *oletetun* keskimääräisen viiveen jokaiselle reititysstrategia x varikkoalue -yhdistelmälle. Kiinnitämme satunnaissiemenen ja lisäämme merkityksettömän hajonnan, jotta keskiarvot näyttävät realistisilta, samalla kun 3x3-tasapainotettu rakenne säilyy. `cell_n`-paino arvolla 1 jokaisessa solussa kertoo GLMPOWER:lle, että asetelma on tasapainotettu.

In [1]:
/* ================================================================
   Generoi esimerkinomainen aineisto oletetuista keskimääräisistä
   viiveistä. Yksi rivi per reititysstrategia x varikkoalue
   -suunnittelusolu.
   ================================================================ */
TIEDOT routing_trial;
   CALL streaminit(20260531);
   PITUUS routing_strategy $8 depot_region $9;
   TAULUKKO strat[3]  $8 _temporary_ ('Static' 'Dynamic' 'AIOpt');
   TAULUKKO region[3] $9 _temporary_ ('Northeast' 'Midwest' 'West');
   TAULUKKO smean[3]     _temporary_ (18.0 14.5 11.0);   /* oletetut strategiakeskiarvot */
   TAULUKKO radj[3]      _temporary_ (1.5  0.0 -1.0);    /* aluekohtaiset korjaukset (min) */
   TEE i = 1 ASTI 3;
      TEE j = 1 ASTI 3;
         routing_strategy = strat[i];
         depot_region     = region[j];
         mean_delay_min   = smean[i] + radj[j]
                            + rand('normal', 0, 0.4);
         cell_n           = 1;
         TULOSTE;
      LOPPU;
   LOPPU;
   POISTA i j;
SUORITA;

PROSEDUURI TULOSTA TIEDOT=routing_trial NIMIKE noobs;
   MUUTTUJA routing_strategy depot_region mean_delay_min cell_n;
   NIMIKE routing_strategy="Reititysstrategia" depot_region="Varikkoalue"
         mean_delay_min="Keskimääräinen viive (min)" cell_n="Solun paino (n)";
   OTSIKKO "Esimerkinomaiset soluarvot: oletettu toimitusviive (minuuttia)";
SUORITA;

                             Esimerkinomaiset soluarvot: oletettu toimitusviive (minuuttia)                             

Reititysstrategia  Varikkoalue     Keskimääräinen viive (min)  Solun paino (n)
Static             Northeast                     19.687507651                1
Static             Midwest                      17.9871067398                1
Static             West                         16.8240210086                1
Dynamic            Northeast                    15.9537535365                1
Dynamic            Midwest                      14.4415169858                1
Dynamic            West                         12.8636389493                1
AIOpt              Northeast                    12.6143811724                1
AIOpt              Midwest                       10.893885771                1
AIOpt              West                           9.635351403                1




NOTE: DATA routing_trial


NOTE: Wrote routing_trial (9 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=routing_trial

NOTE: PROC PRINT completed: 9 observations printed, 4 variables


---
## 2. Otoskoko kokonaisasetelmalle

Asetelman ollessa valmiina pyydämme GLMPOWER:ia **ratkaisemaan kokonaisotoskoon** (`NTOTAL = .`) 80% ja 90% voima-arvolla. `MODEL`-lauseke määrittää kaksisuuntaisen asetelman yhdysvaikutuksineen (`routing_strategy*depot_region`), `WEIGHT cell_n` ilmoittaa tasapainotetut profiilipainot, ja `STDDEV = 8` on toimitusviiveen oletettu jäännöskeskivirhe. `NFRACTIONAL` antaa proseduurin raportoida tarkat murtolukumäärät ennen ylöspäin pyöristystä.

Esirekisteröimme myös kolme suunniteltua `CONTRAST`-vertailua -- AI-optimoitu vs. staattinen, dynaaminen vs. staattinen, ja mikä tahansa uudelleenreititys vs. staattinen -- jotka dokumentoivat toiminnallisesti merkitykselliset hypoteesit, joita varten koe on rakennettu.

In [2]:
/* ================================================================
   Ratkaise kuljettajien kokonaismäärä, joka tarvitaan 80% ja 90%
   voima-arvon saavuttamiseksi reititysstrategia- ja
   varikkoaluevaikutuksille.
   ================================================================ */
PROSEDUURI glmpower TIEDOT=routing_trial;
   LUOKKA routing_strategy depot_region;
   MODEL mean_delay_min = routing_strategy depot_region routing_strategy*depot_region;
   PAINO cell_n;
   CONTRAST "AI-optimoitu vs. staattinen"      routing_strategy -1  0  1;
   CONTRAST "Dynaaminen vs. staattinen"        routing_strategy -1  1  0;
   CONTRAST "Mikä tahansa uudelleenreititys vs. staattinen" routing_strategy -2  1  1;
   NIMIKE routing_strategy="Reititysstrategia" depot_region="Varikkoalue"
         mean_delay_min="Keskimääräinen viive (min)";
   POWER
      nfractional
      STDDEV = 8.0
      ALPHA  = 0.05
      NTOTAL = .
      POWER  = 0.80 0.90;
   OTSIKKO "Otoskoko reititysstrategian vaikutusten havaitsemiseksi viiveessä";
SUORITA;

                             Esimerkinomaiset soluarvot: oletettu toimitusviive (minuuttia)                             

The GLMPOWER Procedure


             Fixed Scenario Elements             

Item                Value                        
------------------  -----------------------------
Dependent Variable  Keskimääräinen viive (min)   
Source              Reititysstrategia            
Source              Varikkoalue                  
Source              Reititysstrategia*Varikkoalue

                  Computed N Total                  

   Alpha   Std Dev   N Total     Power  Actual Power
--------  --------  --------  --------  ------------
  0.0500    8.0000        63    0.8000        0.8035
  0.0500    8.0000        83    0.9000        0.9014





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 3. Voima-arvon herkkyys vaihtelulle ja kokeen koolle

Otoskokovastaus riippuu oletetusta virheen keskihajonnasta, jota harvoin tunnetaan tarkasti etukäteen. Tässä käännämme kysymyksen: **kiinnitämme** useita ehdokas-ilmoittautumismääriä (`NTOTAL = 45 90 180`) ja **ratkaisemme saavutetun voima-arvon** (`POWER = .`) uskottavien viiveen keskihajontojen (6, 8 ja 10 minuuttia) ruudukossa. Syntyvä taulukko on herkkyyskartta -- se näyttää, kuinka vankka kukin ilmoittautumissuunnitelma on, jos todellisen maailman reittivaihtelu osoittautuu toivottua suuremmaksi.

In [3]:
/* ================================================================
   Herkkyysruudukko: saavutettu voima-arvo ehdokaskokojen ja
   uskottavien viiveen keskihajontojen yli.
   ================================================================ */
PROSEDUURI glmpower TIEDOT=routing_trial;
   LUOKKA routing_strategy depot_region;
   MODEL mean_delay_min = routing_strategy depot_region;
   PAINO cell_n;
   NIMIKE routing_strategy="Reititysstrategia" depot_region="Varikkoalue"
         mean_delay_min="Keskimääräinen viive (min)";
   POWER
      nfractional
      STDDEV = 6.0 8.0 10.0
      ALPHA  = 0.05
      NTOTAL = 45 90 180
      POWER  = .;
   OTSIKKO "Saavutettu voima-arvo vaihtelu- ja koekokoskenaarioissa";
SUORITA;

                             Esimerkinomaiset soluarvot: oletettu toimitusviive (minuuttia)                             

The GLMPOWER Procedure


             Fixed Scenario Elements             

Item                Value                        
------------------  -----------------------------
Dependent Variable  Keskimääräinen viive (min)   
Source              Reititysstrategia            
Source              Varikkoalue                  

            Computed Power            

   Alpha   Std Dev   N Total     Power
--------  --------  --------  --------
  0.0500    6.0000        45    0.8084
  0.0500    8.0000        45    0.5475
  0.0500   10.0000        45    0.3729
  0.0500    6.0000        90    0.9868
  0.0500    8.0000        90    0.8681
  0.0500   10.0000        90    0.6782
  0.0500    6.0000       180    1.0000
  0.0500    8.0000       180    0.9943
  0.0500   10.0000       180    0.9434





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 4. Voimakäyrä ilmoittautumispäätöstä varten

Lopuksi piirrämme **voimakäyrän** -- saavutetun voima-arvon kuljettajien kokonaisilmoittautumisen kasvaessa 30:stä 270:een 30:n askelin -- strategia-plus-alue-mallille odotetulla 8 minuutin keskihajonnalla. `POWER = .`:n ratkaiseminen tuon ilmoittautumisruudukon yli tuottaa käyrän taulukoituna `N Total` vs. `Power`-sarjana, joten voimme lukea, missä ilmoittautumismäärässä kumpikin tavanomainen 80% ja 90% tavoite ylittyy.

In [4]:
/* ================================================================
   Voimakäyrä: saavutettu voima-arvo vs. ilmoittautuneiden
   kuljettajien kokonaismäärä, käyden läpi 30:stä 270:een 30:n
   askelin odotetulla 8 minuutin keskihajonnalla.
   ================================================================ */
PROSEDUURI glmpower TIEDOT=routing_trial;
   LUOKKA routing_strategy depot_region;
   MODEL mean_delay_min = routing_strategy depot_region;
   PAINO cell_n;
   NIMIKE routing_strategy="Reititysstrategia" depot_region="Varikkoalue"
         mean_delay_min="Keskimääräinen viive (min)";
   POWER
      nfractional
      STDDEV = 8.0
      ALPHA  = 0.05
      NTOTAL = 30 60 90 120 150 180 210 240 270
      POWER  = .;
   OTSIKKO "Voimakäyrä: saavutettu voima-arvo vs. kuljettajien kokonaismäärä";
SUORITA;

                             Esimerkinomaiset soluarvot: oletettu toimitusviive (minuuttia)                             

The GLMPOWER Procedure


             Fixed Scenario Elements             

Item                Value                        
------------------  -----------------------------
Dependent Variable  Keskimääräinen viive (min)   
Source              Reititysstrategia            
Source              Varikkoalue                  

            Computed Power            

   Alpha   Std Dev   N Total     Power
--------  --------  --------  --------
  0.0500    8.0000        30    0.3733
  0.0500    8.0000        60    0.6887
  0.0500    8.0000        90    0.8681
  0.0500    8.0000       120    0.9500
  0.0500    8.0000       150    0.9826
  0.0500    8.0000       180    0.9943
  0.0500    8.0000       210    0.9982
  0.0500    8.0000       240    0.9995
  0.0500    8.0000       270    0.9999





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 5. Tulkinta ja suositus

Analyysi antaa toimintotiimille perustellun ilmoittautumissuunnitelman:

- **Perusmitoitus (Osa 2).** Olettaen 8 minuutin jäännöskeskihajonnan, täysi kaksisuuntainen malli (strategia, alue ja niiden yhdysvaikutus) saavuttaa **80% voima-arvon 63 kuljettajalla** ja **90% voima-arvon 83 kuljettajalla**. Pyöristäen ylöspäin poistuman varalta, noin **90 kuljettajan** ilmoittautumistavoite ylittää 90%:n kynnyksen mukavasti.

- **Herkkyydellä on väliä (Osa 3).** Voima-arvo on erittäin herkkä viiveen vaihtelulle. 90 kuljettajalla saavutettu voima-arvo laskee ~99%:sta (SD=6) ~87%:iin (SD=8) ja ~68%:iin (SD=10). 45 kuljettajan pilotti riittää vain, jos vaihtelu on vähäistä (~81% voima-arvo SD=6:lla), ja se on pahasti alimitoitettu (~37%) jos SD nousee arvoon 10. Käytännön johtopäätös: johdonmukaisiin, hyvin instrumentoituihin reitteihin panostaminen vaihtelun hillitsemiseksi on yhtä arvokasta kuin kuljettajien lisääminen.

- **Voimakäyrä (Osa 4).** Piirrettynä strategia-plus-alue-mallille (ilman yhdysvaikutustermiä, herkkyystarkastelussa käytetty näkökulma), saavutettu voima-arvo nousee arvosta 0.37 30 kuljettajalla arvoon 0.69 60:llä, 0.87 90:llä ja 0.95 120:llä, tasoittuen yli 0.99:n 180:lla. Käyrää tavoitteita vasten luettaessa 80% voima-arvo saavutetaan noin 77 kuljettajalla ja 90% noin 99:llä -- hieman korkeampi kuin täyden mallin mitoitus Osassa 2, koska yhdysvaikutustermin poistaminen jakaa saman vaikutuksen harvemmille mallin vapausasteille.

**Suositus:** Ilmoittaudu noin **90 kuljettajaa** (30 per reititysstrategia, tasapainotettuna kolmen varikkoalueen kesken). Tämä ylittää 90%:n voima-arvon täydelle mallille odotetulla 8 minuutin keskihajonnalla, pitää ~87%:n voima-arvon jopa konservatiivisemmalla supistetun mallin käyrällä, ja pitää kokeen tarpeeksi pienenä toteutettavaksi yhden toimintavuosineljänneksen sisällä.

*Huomautus:* GLMPOWER analysoi oletetun *asetelman*, ei havaittuja tuloksia -- joten näiden lukujen uskottavuus perustuu oletettuihin keskiarvoihin ja keskihajontaan. Tiimien tulisi tarkistaa mitoitus, kun lyhyt pilotti tuottaa empiirisen arvion toimitusviiveen vaihtelusta.